# EXIST 2026 — Task 3 (vídeos) — braço CLIP, FASE 2: fine-tune com heads A/B congeladas

**⚠ CONTINUAÇÃO DE PESOS, NÃO RE-TREINO.** Este notebook parte do checkpoint final da
run `run_B2_bio_viewer_seed42_CLIP` (15 épocas, braço CLIP oficial) e treina **mais 5
épocas**. Pré-registro completo em `improvements/22_HEAD_FREEZE_PREREGISTRATION.md` —
ler antes de mexer em qualquer valor deste notebook.

**Hipótese (4ª testada neste braço):** as heads A/B disputam o trunk com a head C;
congelar A/B libera capacidade para C sem que A/B degradem.

**Dois braços, um eixo de diferença:**

| `FREEZE_AB` | ARM | `head_a`/`head_b` | papel |
|---|---|---|---|
| `True`  | `CLIP_FT5_FREEZEAB` | congeladas | braço da hipótese |
| `False` | `CLIP_FT5_CTRL`     | treinam    | **controle de orçamento** |

Rodar o notebook **duas vezes**, uma com cada valor de `FREEZE_AB` (célula 2.1). Sem o
controle, "freeze ajudou" seria inseparável de "5 épocas extras ajudaram" — dois eixos
numa run só, proibido pela regra do doc 18 §9.5.

**A loss de A e B continua ATIVA, com os pesos originais (1,0 / 0,7 / 0,7).**
`requires_grad=False` numa head impede que os PESOS dela mudem, mas o gradiente continua
fluindo através dela até o trunk — as heads congeladas viram probes fixas que ANCORAM o
trunk. Remover as losses de A/B é o que soltaria o trunk. Não "otimizar" isso; ver
`vcc.set_heads_trainable` e doc 22 §1.1.

⚠ A repesagem `LOSS_W_C=4,2` do braço WC42 **não** se aplica aqui (foi refutada, doc 18
§9.5, e seria um segundo eixo). A célula 11 assert-a `vc.LOSS_W_C == 0.7`.

⚠ O estado do AdamW e o scheduler **nascem do zero** (só pesos treináveis foram salvos):
warmup e cosine recomeçam nas 5 épocas. É continuação de pesos, não resume de treino.

⚠ Estado reportado: **FINAL da época 5** nos dois braços (mesma regra de simetria do
projeto). O `RobustScaler` vem do checkpoint-base, nunca re-ajustado (BUG-B/ARCH-18).

## 1. Imports

Nada de lógica aqui — só os módulos e o mínimo para orquestrar. `video_core` fornece o
split congelado, o dataset base, o `Trainer` multi-task e as métricas; `video_core_clip`
fornece só o que muda para o encoder CLIP (dataset renormalizado, modelo, otimizador,
save/load, selfchecks).

In [1]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


cwd: /home/a67606/Thesis/EXIST_VIDEOS


In [2]:
import json

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler
from transformers import TrainingArguments, XLMRobertaTokenizer

import video_core as vc
import video_core_clip as vcc


## 2. Configuração da run

**Braço único: `ARM = "CLIP"`.** Diferente de `Thesis_video_improved.ipynb` (que
seleciona entre CAL/A_prime/A/B/B15 na mesma família VideoMAE), este notebook só tem um
braço — a substituição de encoder é a variável em teste, não a atribuição de features.

- `SPLIT_SEED` é **fixo** e vive em `video_core` — o mesmo split congelado de todas as
  outras runs.
- `INIT_SEED` = 42 por padrão (única seed rodada para este braço de atribuição isolado;
  não faz parte da tabela de ablação cross-seed).

In [3]:
INIT_SEED      = 42        # mesma seed da fase 1
USE_BIO        = True      # False -> fusion_input_dim_clip(False) = 2816
USE_VIEWER_EMB = True

CONDITION = vc.condition_tag(USE_BIO, USE_VIEWER_EMB)

# NUM_EPOCHS, ARM, RUN_DIR, CKPT_DIR, FREEZE_AB e BASE_RUN_DIR sao definidos na secao
# 2.1, que e a UNICA fonte deles. Nao redefinir aqui.
print(f"condição = {CONDITION} | INIT_SEED = {INIT_SEED} | SPLIT_SEED = {vc.SPLIT_SEED}")
print(f"max_length = {vc.MAX_LENGTH}")

condição = B2_bio_viewer | INIT_SEED = 42 | SPLIT_SEED = 42
max_length = 256


## 2.1 Configuração da fase 2

`FREEZE_AB` é a **única** coisa a mudar entre as duas execuções. Tudo o mais é idêntico
por construção: mesmo checkpoint-base, mesmas 5 épocas, mesmas seeds, mesma
`TrainingArguments`, mesmo scaler.

`NUM_EPOCHS_FT = 5` está **fixado a priori** (doc 22 §2). Escolher a duração depois de
ver a curva é o leakage que o projeto proíbe.

In [ ]:
from pathlib import Path

# ⚠ ÚNICO interruptor entre os dois braços. Rodar o notebook 2x: True, depois False.
FREEZE_AB = False

FROZEN_HEADS = ("head_a", "head_b") if FREEZE_AB else ()
ARM = "CLIP_FT5_FREEZEAB" if FREEZE_AB else "CLIP_FT5_CTRL"

# Checkpoint-base: braço CLIP oficial (15 épocas, estado final). NÃO o WC42 — a
# repesagem de LOSS_W_C foi refutada (doc 18 §9.5) e seria um segundo eixo.
BASE_RUN_DIR = Path("run_B2_bio_viewer_seed42_CLIP")
assert BASE_RUN_DIR.exists(), f"checkpoint-base ausente: {BASE_RUN_DIR}"

AUDIO_VERSION = "v2"
NUM_EPOCHS = 5          # NUM_EPOCHS_FT — fixado A PRIORI, doc 22 §2. Não ajustar depois
                        # de ver a curva.

RUN_DIR  = Path(f"{vc.run_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")
CKPT_DIR = Path(f"{vc.ckpt_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")
assert RUN_DIR != BASE_RUN_DIR, "RUN_DIR não pode sobrescrever o checkpoint-base"

print(f"ARM = {ARM} | FREEZE_AB = {FREEZE_AB} | congeladas = {list(FROZEN_HEADS) or 'nenhuma'}")
print(f"base = {BASE_RUN_DIR} (15 épocas) -> +{NUM_EPOCHS} épocas de fase 2")
print(f"RUN_DIR  = {RUN_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")

ARM = CLIP_FT5_FREEZEAB | FREEZE_AB = True | congeladas = ['head_a', 'head_b']
base = run_B2_bio_viewer_seed42_CLIP (15 épocas) -> +5 épocas de fase 2
RUN_DIR  = run_B2_bio_viewer_seed42_CLIP_FT5_FREEZEAB
CKPT_DIR = checkpoints_B2_bio_viewer_seed42_CLIP_FT5_FREEZEAB


## 3. Determinismo e self-checks

`set_all_seeds` é chamado de novo imediatamente antes de cada init de pesos — é o que
`vcc.build_model_clip` faz internamente (mesmo protocolo do `vc.build_model`: as
matrizes LoRA A do XLM-R e as camadas destravadas do CLIP nascem de RNG explicitamente
re-semeado, não do `set_seed` tardio do Trainer).

`vc.run_selfchecks()` continua válido aqui — cobre convenções da Task A, taxa de
modality dropout, etc., que não mudam com o encoder de vídeo.

In [5]:
vc.set_all_seeds(INIT_SEED)
vc.run_selfchecks()


✓ convenção Task A (sigmoid = P(YES), eixo [NO, YES]) verificada
✓ fusion_input_dim: 2688 com bio (B1/B2), 2560 sem bio (B0)
✓ modality dropout: taxas por modalidade ≈ ['0.0503', '0.0495', '0.0499', '0.0501'] (p=0.05); B0 apenas não usa a 4ª faixa, as outras três não mudam


## 4. Dados e split **congelado**

Mesmo split congelado (`Dataset/val_split_42.json`) de todas as outras runs — o braço
CLIP compartilha o mesmo val, para que qualquer comparação futura (mesmo que informal,
já que o braço é incomparável por state_dict) ao menos use os mesmos itens de avaliação.

⚠ **O treino roda numa VM separada e `Dataset/` não é sincronizado.** Copiar
`Dataset/val_split_42.json` para a VM antes da primeira run: a célula abaixo tem de
imprimir `[split] carregado de ...`. Se imprimir `[split] artefato gravado`, a run está
**fora de protocolo**.

In [6]:
assert vc.split_artifact_path().exists(), (
    "Copiar Dataset/val_split_42.json para a VM antes de rodar. Sem ele, "
    "build_frozen_split RECALCULA o split em vez de carregar o congelado.")

full_df = pd.read_json("./Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
print(f"Treino: {len(train_df)} | Val: {len(val_df)}")


[split] carregado de Dataset/val_split_42.json (seed=42, estratificação=task_a): 2019 treino / 505 val
Treino: 2019 | Val: 505


## 5. Carregar o checkpoint-base (modelo + `RobustScaler`)

`vcc.load_run_clip` devolve os pesos da fase 1 (15 épocas, estado final) e o
`RobustScaler` que ela ajustou **só no treino** — re-ajustar aqui seria vazamento
(BUG-B/ARCH-18). O fingerprint do estado carregado vai para o `meta.json` como
proveniência da fase 2.

In [7]:
# Fase 2 carrega modelo E scaler do checkpoint-base. O RobustScaler NUNCA é
# re-ajustado (BUG-B/ARCH-18) — é o mesmo objeto que a fase 1 usou.
model, scaler = vcc.load_run_clip(BASE_RUN_DIR, init_seed=INIT_SEED)
assert scaler is not None, f"{BASE_RUN_DIR}/scaler.pkl ausente — sem ele não há fase 2"

base_fingerprint = vc.model_fingerprint(model)
print("fingerprint do checkpoint-base:", base_fingerprint)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_a

[build_clip_vision] 4 últimas camadas destravadas, 50,384,896 params treináveis


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[load_run_clip] run_B2_bio_viewer_seed42_CLIP carregado | video_encoder=clip | use_bio=True, use_viewer_emb=True | 718 pesos congelados vindos dos backbones
fingerprint do checkpoint-base: f84b9ff6db733041


## 6. Datasets

`vcc.EXISTVideoDatasetCLIP` herda de `vc.EXISTVideoDataset` — carrega os mesmos frames
v2 em disco (normalizados para VideoMAE/ImageNet) e renormaliza por afim composta para
a estatística do CLIP, sem reler disco. `max_length=256` (ARCH-05), igual ao braço
VideoMAE.

In [8]:
tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)

train_dataset = vcc.EXISTVideoDatasetCLIP(
    train_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)
val_dataset = vcc.EXISTVideoDatasetCLIP(
    val_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


Train: 2019 | Val: 505


## 7. Sanity checks dos dados

Além do shape dos frames (idêntico ao braço VideoMAE — o dataset CLIP consome o mesmo
tensor `[16,3,224,224]`, só muda a estatística de normalização), roda-se aqui
`vcc.selfcheck_renorm_equivalence` — verifica sobre itens reais em disco que a
renormalização afim composta (VideoMAE → CLIP) é aritmeticamente equivalente a
renormalizar a partir do zero, dentro de `atol=1e-5`. Portão só executável na VM (com
frames reais em disco); localmente não tem valor probatório.

In [9]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:20s}: {tuple(v.shape) if torch.is_tensor(v) else v}")

assert sample["frames"].shape == (vc.NUM_FRAMES, vc.NUM_CHANNELS, vc.FRAME_H, vc.FRAME_W)
print("✓ shape dos frames confere (renormalizados para a estatística do CLIP)")

print("'labels' no batch?", "labels" in sample, "(tem de ser False)")

cvpc_values = [train_dataset[i]["cvpc"].item() for i in range(len(train_dataset))]
print(f"CVPC mean={np.mean(cvpc_values):.4f} std={np.std(cvpc_values):.4f} "
      f"range=[{min(cvpc_values):.4f}, {max(cvpc_values):.4f}]")

# Portão: equivalência da renormalização afim composta contra recomputar do zero.
vcc.selfcheck_renorm_equivalence(train_dataset)
print("✓ selfcheck_renorm_equivalence passou")


id_EXIST            : 120001
frames              : (16, 3, 224, 224)
audio               : (1280,)
input_ids           : (256,)
attention_mask      : (256,)
bio_vec             : (16,)
cvpc                : (1,)
user_ids            : (2,)
labels_a            : ()
labels_b            : (3,)
labels_c            : (6,)
✓ shape dos frames confere (renormalizados para a estatística do CLIP)
'labels' no batch? False (tem de ser False)
CVPC mean=0.0883 std=0.3861 range=[-0.8384, 1.0000]
✓ selfcheck_renorm_equivalence: 5 itens, max|diff|=2.38e-07 < atol=1e-05
✓ selfcheck_renorm_equivalence passou


## 8. Estado carregado + freeze das heads

Nada é construído do zero: o modelo veio do checkpoint-base na seção 5. Aqui só se
confere a largura de fusão e se aplica o freeze pré-registrado (`FROZEN_HEADS`).

⚠ `requires_grad=False` numa head impede que os **pesos dela** mudem, mas o gradiente
continua fluindo **através** dela até `fusion` e o trunk. As heads A/B congeladas viram
probes fixas cuja loss **ancora** o trunk. Remover as losses de A/B é o que soltaria o
trunk — não fazer isso. Doc 22 §1.1.

In [10]:
# O modelo JÁ veio de vcc.load_run_clip (seção 5) — nada é construído do zero aqui.
expected_fusion_in = vcc.fusion_input_dim_clip(USE_BIO)
assert model.fusion_in == expected_fusion_in, (
    f"fusion_in={model.fusion_in} != esperado {expected_fusion_in}")
print("fusion_in:", model.fusion_in)
print("params:", vc.count_params(model))

# Freeze das heads pré-registradas (doc 22). ⚠ NÃO corta o gradiente do trunk: a loss de
# A/B continua fluindo através das heads congeladas e ANCORA o trunk — é esse o
# mecanismo pretendido. Ver docstring de vcc.set_heads_trainable.
HEAD_PARAMS = vcc.set_heads_trainable(model, frozen=FROZEN_HEADS)
vcc.assert_heads_frozen(model, FROZEN_HEADS)

vc.record_fingerprint(RUN_DIR, "init", model)

fusion_in: 2944
params: {'total': 868805339, 'trainable': 56120027, 'trainable_pct': 6.4594477589875865}
[heads] congeladas=['head_a', 'head_b'] | head_a:0 head_b:0 head_c:6,150  (params treináveis por head)
✓ freeze confere: congeladas=['head_a', 'head_b']
[fingerprint] init: f84b9ff6db733041  (primeira run; nada a comparar)


'f84b9ff6db733041'

## 8.1 Smoke test — PORTÃO DURO

Mesmo portão da fase 1 (`vcc.smoke_test_clip`: 1 forward + backward + step sob bf16 e
gradient checkpointing, memória de pico, timing extrapolado, grads não-nulos nas 4
camadas destravadas do CLIP).

⚠ **Diferença crítica na fase 2:** o smoke test dá um `optimizer.step()` REAL e portanto
**muta os pesos carregados** — e muta diferente nos dois braços (com A/B congeladas, as
heads não recebem o passo). Deixar isso passar faria `FREEZE_AB` e `CTRL` partirem de
pontos diferentes, destruindo o controle. A célula abaixo **recarrega o checkpoint-base**
logo após o smoke test e prova a restauração por fingerprint.

In [11]:
vcc.smoke_test_clip(model, train_dataset, device="cuda", micro_bs=2)
print("✓ smoke_test_clip passou")

# ⚠ OBRIGATÓRIO NA FASE 2: smoke_test_clip faz um optimizer.step() real — ele MUTA os
# pesos carregados. Pior: muta DIFERENTE nos dois braços (com A/B congeladas as heads não
# recebem o passo), o que faria FREEZE_AB e CTRL partirem de pontos distintos e
# destruiria o controle. Recarregar o checkpoint-base restaura o ponto de partida exato;
# o assert de fingerprint prova que restaurou.
import safetensors.torch as _st
_sd = _st.load_file(str(BASE_RUN_DIR / "trainable.safetensors"), device="cpu")
model.load_state_dict(_sd, strict=False)
del _sd
restored = vc.model_fingerprint(model)
assert restored == base_fingerprint, (
    f"fingerprint pós-restauração {restored} != base {base_fingerprint} — o ponto de "
    f"partida da fase 2 não é o checkpoint-base; NÃO treinar assim.")
print(f"✓ pesos do checkpoint-base restaurados após o smoke test ({restored})")

# O freeze é reaplicado porque load_state_dict não mexe em requires_grad, mas o
# smoke test criou um otimizador próprio — reafirmar o estado pré-registrado é barato.
vcc.set_heads_trainable(model, frozen=FROZEN_HEADS)
vcc.assert_heads_frozen(model, FROZEN_HEADS)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


─── smoke_test_clip ───
  loss             : 0.6639
  fusion_in        : 2944
  micro_bs         : 2
  tempo (1 step)   : 1.094 s
  memória pico     : 4193.3 MB
  timing p/ época  : ~1103.3 s (1009 passos, extrapolado)
  grads não-nulos  : 64/64 params das 4 camadas destravadas do CLIP
✓ smoke_test_clip passou
✓ pesos do checkpoint-base restaurados após o smoke test (f84b9ff6db733041)
[heads] congeladas=['head_a', 'head_b'] | head_a:0 head_b:0 head_c:6,150  (params treináveis por head)
✓ freeze confere: congeladas=['head_a', 'head_b']


## 9. `TrainingArguments`

Idêntica à da fase 1 (bs 2, accum 16, bf16, gradient checkpointing), **exceto
`num_train_epochs = 5`**. Manter o resto igual é o que faz da fase 2 uma continuação
comparável, e não um regime de treino novo.

⚠ **Warmup e cosine recomeçam do zero.** Só os pesos treináveis foram salvos na fase 1 —
o estado do AdamW e o scheduler nascem novos aqui. É continuação de PESOS, não resume de
otimização (doc 22 §1.2). Vale igual nos dois braços, logo não afeta a comparação
`FREEZE_AB` vs `CTRL`.

- `warmup_steps=0.1` (float `[0,1)` = ratio) — mesma escolha da fase 1.
- `load_best_model_at_end=False`, `metric_for_best_model` ausente — selecionar época pela
  métrica reportada é leakage.
- **`save_strategy="no"`** — o modelo final é salvo à mão na seção 13. O
  `SaveTrainableStateCallback` (seção 10) grava o estado por época só como registro
  diagnóstico, e é removido na seção 14 depois da validação.

In [12]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=NUM_EPOCHS,

    # ── batch: IDÊNTICA à congelada do braço VideoMAE (comparabilidade de custo) ──
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,          # batch efetivo = 32
    gradient_checkpointing=True,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    dataloader_pin_memory=True,

    eval_strategy="epoch",
    save_strategy="no",                      # checkpoint completo do HF é caro demais
    # (backbone congelado + otimizador). Estado por época salvo enxuto via callback
    # (seção 10) — ver célula markdown acima.
    load_best_model_at_end=vc.LOAD_BEST_MODEL_AT_END,   # False — não ligar (leakage)
    # metric_for_best_model: deliberadamente AUSENTE. Ver markdown acima.

    warmup_steps=0.1,                        # float ∈ [0,1) = ratio (durabilidade, não bugfix)
    lr_scheduler_type="cosine",

    remove_unused_columns=False,
    label_names=["labels_a", "labels_b", "labels_c"],

    report_to="none",
    logging_steps=10,
    seed=INIT_SEED,
    data_seed=INIT_SEED,
)


## 10. Trainer

`data_collator=vc.video_collate` é **obrigatório**, mesmo motivo do braço VideoMAE:
o dataset devolve `id_EXIST` (str).

O otimizador é `vcc.get_optimizer_clip(model)` — não o `vc.get_optimizer(model)` default.
Tem um grupo dedicado para os parâmetros `videomae.*` (que aqui são os do CLIP, wart de
nomenclatura documentada) com `lr=CLIP_LR=1e-5`, verificado **primeiro** no predicado
antes de cair no `base_decay` (1e-3) — cair lá destruiria os pesos pré-treinados do
ViT-L sem erro nenhum. `get_optimizer_clip` já roda dois asserts internos (grupo CLIP
com a contagem esperada de params p/ `UNFROZEN_VISION_LAYERS`; partição exata) antes de
devolver o otimizador.

**`SaveTrainableStateCallback` (registro diagnóstico, não usado pra seleção):** salva
`vc.trainable_state_dict(model)` — só os params treináveis — em
`CKPT_DIR/epoch_states/epoch_NN.safetensors` a cada `on_evaluate`. A run oficial é o
estado FINAL da época 15 (decisão da célula 0/2.1); estes arquivos existem só para
permitir reconstruir a curva de qualquer época sem re-treinar, e são removidos na
seção 14 depois que a run oficial é validada.

`trainer.label_names` é setado também depois da construção — padrão idêntico ao braço
VideoMAE, copiado literalmente (é a forma empiricamente conhecida por funcionar em
`transformers==5.6.0`).

In [13]:
from transformers import TrainerCallback
from safetensors.torch import save_file as st_save_file
from accelerate.utils import extract_model_from_parallel

EPOCH_STATES_DIR = CKPT_DIR / "epoch_states"

class SaveTrainableStateCallback(TrainerCallback):
    """Salva vc.trainable_state_dict(model) — só os params TREINÁVEIS, ~220MB — a
    cada avaliação (dispara junto com eval_strategy="epoch", 1x/época). Substitui o
    save_strategy="epoch" do HF Trainer, que salvaria o checkpoint completo (backbone
    congelado + otimizador) e arriscaria esgotar disco em 15 épocas — ver célula
    markdown da seção 9."""

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        m = extract_model_from_parallel(model if model is not None else kwargs["model"])
        epoch_int = round(state.epoch)
        EPOCH_STATES_DIR.mkdir(parents=True, exist_ok=True)
        path = EPOCH_STATES_DIR / f"epoch_{epoch_int:02d}.safetensors"
        sd = vc.trainable_state_dict(m)
        # Heads congeladas não entram em trainable_state_dict (filtra por requires_grad)
        # — incluí-las explicitamente mantém o estado por época recarregável.
        full = m.state_dict()
        for h in vcc.FREEZABLE_HEADS:
            for k in full:
                if k.startswith(h + "."):
                    sd[k] = full[k].detach().cpu().clone()
        st_save_file(sd, str(path))
        print(f"[epoch_states] época {epoch_int} salva em {path.name}")

epoch_state_callback = SaveTrainableStateCallback()

optimizer = vcc.get_optimizer_clip(model)

trainer = vc.MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=vc.compute_metrics,
    data_collator=vc.video_collate,
    optimizers=(optimizer, None),
    callbacks=[epoch_state_callback],
)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]

print("model_accepts_loss_kwargs:", trainer.model_accepts_loss_kwargs, "(travado em False)")
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size
               * training_args.gradient_accumulation_steps)) * NUM_EPOCHS
print("warmup steps:", training_args.get_warmup_steps(total_steps),
      f"de ~{total_steps} steps totais")


model_accepts_loss_kwargs: False (travado em False)
warmup steps: 32 de ~315 steps totais


## 11. Treino

In [14]:
# Portões do doc 22 §4 — falham duro, não avisam.
assert vc.LOSS_W_C == 0.7, (
    f"LOSS_W_C={vc.LOSS_W_C} — a fase 2 roda com os pesos ORIGINAIS (1,0/0,7/0,7). "
    f"A repesagem 4,2 do braço WC42 foi refutada (doc 18 §9.5) e seria um 2º eixo.")
vcc.assert_heads_frozen(model, FROZEN_HEADS)
print(f"[pré-treino] LOSS_W_A={vc.LOSS_W_A} LOSS_W_B={vc.LOSS_W_B} LOSS_W_C={vc.LOSS_W_C} "
      f"| congeladas={list(FROZEN_HEADS) or 'nenhuma'} | épocas={NUM_EPOCHS}")

train_output = trainer.train()
print(train_output)

✓ freeze confere: congeladas=['head_a', 'head_b']
[pré-treino] LOSS_W_A=1.0 LOSS_W_B=0.7 LOSS_W_C=0.7 | congeladas=['head_a', 'head_b'] | épocas=5


Epoch,Training Loss,Validation Loss,Task A Icm Hard,Task A Icmnorm Hard,Task A Fmeasure Hard,Task A Icmsoft Soft,Task A Icmsoftnorm Soft,Task A Crossentropy Soft,Task B Icm Hard,Task B Icmnorm Hard,Task B Fmeasure Hard,Task B Icmsoft Soft,Task B Icmsoftnorm Soft,Task B Crossentropy Soft,Task C Icmsoft Soft,Task C Icmsoftnorm Soft,Task C Icm Hard,Task C Icmnorm Hard,Task C Fmeasure Hard,Proxy Task A F1,Proxy Task A Auc,Proxy Task B F1,Proxy Task C F1
1,0.479706,2.060252,0.130718,0.565374,0.709156,-0.023339,0.495899,1.244018,-0.024454,0.491565,nan,-0.925556,0.393266,nan,-7.131663,0.127715,-0.649761,0.342728,nan,0.709156,0.759696,0.609539,0.296018
2,0.535948,1.727030,0.082877,0.541448,0.692609,-0.186550,0.467220,1.052227,-0.085259,0.470591,nan,-1.236475,0.357412,nan,-7.639854,0.101187,-0.706810,0.328920,nan,0.692609,0.767682,0.600273,0.304114
3,0.423157,1.938335,0.076578,0.538298,0.689780,-0.105750,0.481418,1.155686,-0.072619,0.474951,nan,-0.961020,0.389177,nan,-6.961766,0.136584,-0.695682,0.331613,nan,0.689780,0.762944,0.598739,0.279084
4,0.440548,1.958276,0.131329,0.565679,0.710062,-0.059568,0.489533,1.182574,0.017160,0.505919,nan,-0.896238,0.396647,nan,-7.154489,0.126524,-0.620631,0.349779,nan,0.710062,0.767235,0.624685,0.330110
5,0.367292,2.025336,0.131431,0.565730,0.710181,-0.013623,0.497606,1.225193,0.040366,0.513924,nan,-0.812552,0.406298,nan,-6.897227,0.139953,-0.625560,0.348586,nan,0.710181,0.768788,0.628662,0.315130


2026-08-16 17:27:59,791 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-16 17:27:59,843 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:27:59,952 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-16 17:27:59,953 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:28:00,075 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:28:00,196 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-16 17:28:00,310 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-16 17:28:00,399 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

### 11.1 Portão pós-treino — o freeze de facto segurou?

Verificação direta **nos pesos**, não nas flags `requires_grad`: no braço `FREEZE_AB` as
heads A/B têm de estar bit a bit idênticas ao checkpoint-base; no `CTRL` têm de ter
mudado. Isto é a afirmação central do experimento, testada no artefato — e pega também a
falha inversa (congelamento acidental no controle), que nada mais checa.

In [15]:
from accelerate.utils import extract_model_from_parallel as _extract
import safetensors.torch as _st

_m = _extract(trainer.model)
_base = _st.load_file(str(BASE_RUN_DIR / "trainable.safetensors"), device="cpu")
_sd = _m.state_dict()

for h in ("head_a", "head_b"):
    keys = [k for k in _base if k.startswith(h + ".")]
    assert keys, f"{h} ausente do checkpoint-base — proveniência quebrada"
    same = all(torch.equal(_sd[k].cpu(), _base[k]) for k in keys)
    if FREEZE_AB:
        assert same, (f"{h} MOVEU durante o treino — o freeze não segurou; "
                      f"resultado deste braço é inválido")
        print(f"✓ {h} bit a bit idêntica ao base (freeze segurou)")
    else:
        assert not same, (f"{h} NÃO moveu no braço CTRL — algo congelou por engano; "
                          f"o controle não é um controle")
        print(f"✓ {h} mudou (braço CTRL, esperado)")

# head_c tem de ter mudado nos DOIS braços — é ela que a fase 2 treina.
_c_keys = [k for k in _base if k.startswith("head_c.")]
assert not all(torch.equal(_sd[k].cpu(), _base[k]) for k in _c_keys), (
    "head_c não mudou — a fase 2 não treinou nada")
print("✓ head_c mudou (treinada nos dois braços, como esperado)")

del _base, _sd, _m

✓ head_a bit a bit idêntica ao base (freeze segurou)
✓ head_b bit a bit idêntica ao base (freeze segurou)
✓ head_c mudou (treinada nos dois braços, como esperado)


In [16]:
# Por que FMeasure/CrossEntropy vêm NaN nas tasks B e C — mesmo comportamento
# documentado no braço VideoMAE, não é específico do encoder CLIP.
import numpy as np, tempfile
from pyevall.evaluation import PyEvALLEvaluation
import video_core as vc

pred_out = trainer.predict(val_dataset)
_, logits_b, _ = vc._unpack_predictions(pred_out.predictions)
_, labels_b, _ = pred_out.label_ids
logits_b, labels_b = np.asarray(logits_b), np.asarray(labels_b)
mask = labels_b[:, 0] != vc.SENTINEL_BC
x = logits_b[mask]; x = x - x.max(axis=-1, keepdims=True)
probs = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
ids = [str(i) for i in range(len(probs))]

with tempfile.TemporaryDirectory() as td:
    ml = vc._metric_list('2_2', 'hard')
    preds = vc._format_for_pyevall(probs, ids, '2_2', 'hard')
    gold = vc._format_for_pyevall(labels_b[mask], ids, '2_2', 'hard')
    import json, os
    pp, gp = f"{td}/p.json", f"{td}/g.json"
    json.dump(preds, open(pp, "w")); json.dump(gold, open(gp, "w"))
    from pyevall.utils.utils import PyEvALLUtils
    rep = PyEvALLEvaluation().evaluate(
        pp, gp, ml, **{PyEvALLUtils.PARAM_HIERARCHY: vc.TASK_2_2_HIERARCHY})
    for m in ml:
        d = rep.report.get('metrics', {}).get(m)
        print(f"  {m}: {'AUSENTE do relatório' if d is None else 'status=' + str(d.get('status'))}")


2026-08-16 17:56:14,567 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-16 17:56:14,609 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:56:14,731 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-16 17:56:14,733 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:56:14,856 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:56:14,973 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-16 17:56:15,095 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-16 17:56:15,182 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

## 12. Curva de validação (registro) — estado oficial é o FINAL da época 5

Nenhum reload acontece aqui. `argmin(eval_loss)` é impresso **só como registro** — a
regra do projeto (simetria de estado final, anti-leakage) vale igual na fase 2, e o
pré-registro do doc 22 §2 travou "estado FINAL da época 5" antes de qualquer curva.

`eval_loss` **é comparável** entre fase 1, `FREEZE_AB` e `CTRL`: mesma escala de loss
(`LOSS_W_C=0,7` nos três) e mesmo conjunto de validação. O que **não** se pode ler dela é
dinâmica de treino — a fase 2 parte de um trunk já com 15 épocas e reinicia warmup e
cosine, então a forma da curva não compara com a da fase 1.

⚠ Não comparar com a `eval_loss` do braço `WC42`: lá o termo de C entra 6× maior
(`LOSS_W_C=4,2`), escala diferente.

In [17]:
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
assert evals, "nenhum eval_loss no log_history — eval_strategy deve estar 'epoch'"

curva = pd.DataFrame(evals)
cols = [c for c in curva.columns
        if c in ("epoch", "step", "eval_loss")
        or c.startswith(("eval_task_a", "eval_task_b", "eval_task_c"))]
print(curva[cols].to_string(index=False))

best = min(evals, key=lambda h: h["eval_loss"])
best_epoch_int = round(best["epoch"])
print(f"\nmin(eval_loss) na época {best_epoch_int} (eval_loss = {best['eval_loss']:.4f}) "
      f"— REGISTRO APENAS. Este braço reporta o estado FINAL da época {NUM_EPOCHS} "
      f"(decisão da célula 0/2.1, ver histórico ali) — nenhum reload acontece aqui.")

# actual_model = exatamente o que o treino deixou (época NUM_EPOCHS), sem
# recarregar nada de epoch_states/. extract_model_from_parallel já foi importado
# na seção 10.
actual_model = extract_model_from_parallel(trainer.model)
print(f"✓ actual_model = estado FINAL da época {NUM_EPOCHS} — este é o estado que "
      f"a seção 13 salva")


 eval_loss  eval_task_a_icm_hard  eval_task_a_icmnorm_hard  eval_task_a_fmeasure_hard  eval_task_a_icmsoft_soft  eval_task_a_icmsoftnorm_soft  eval_task_a_crossentropy_soft  eval_task_b_icm_hard  eval_task_b_icmnorm_hard  eval_task_b_fmeasure_hard  eval_task_b_icmsoft_soft  eval_task_b_icmsoftnorm_soft  eval_task_b_crossentropy_soft  eval_task_c_icmsoft_soft  eval_task_c_icmsoftnorm_soft  eval_task_c_icm_hard  eval_task_c_icmnorm_hard  eval_task_c_fmeasure_hard  epoch  step
  2.060252              0.130718                  0.565374                   0.709156                 -0.023339                      0.495899                       1.244018             -0.024454                  0.491565                        NaN                 -0.925556                      0.393266                            NaN                 -7.131663                      0.127715             -0.649761                  0.342728                        NaN    1.0    64
  1.727030              0.082877          

## 13. Salvar a run (checkpoint enxuto, estado FINAL da época 5)

⚠ **A célula abaixo restaura `requires_grad` de todas as heads ANTES de salvar** — portão
crítico do doc 22 §4.3. `vc.trainable_state_dict` filtra por `requires_grad`, então uma
head congelada não entraria no `trainable.safetensors` e `load_run_clip` **não acusaria**;
o checkpoint recarregaria com essa head em init aleatório, em silêncio.

O `meta.json` da fase 2 grava, além do que a fase 1 já gravava: `phase`, `base_run`,
`base_fingerprint`, `frozen_heads`, `freeze_ab`, `loss_w_a/b/c` e
`total_epochs_incl_phase1` — a identidade destas runs é o freeze mais a proveniência.

In [18]:
# ⚠ PORTÃO CRÍTICO (doc 22 §4.3): restaurar requires_grad de TODAS as heads antes de
# salvar. vc.trainable_state_dict filtra por requires_grad — uma head congelada não
# entraria no trainable.safetensors, e load_run_clip NÃO acusaria (o check `not_loaded`
# de lá só percorre params treináveis). O checkpoint recarregaria com essa head em init
# ALEATÓRIO, em silêncio. A célula de validação da seção 14 é a rede de segurança.
vcc.set_heads_trainable(actual_model)          # frozen=() -> tudo treinável de novo
n_head_params = sum(p.numel() for h in vcc.FREEZABLE_HEADS
                    for p in getattr(actual_model, h).parameters() if p.requires_grad)
print(f"heads restauradas antes do save: {n_head_params:,} params entram no checkpoint")

vcc.save_run_clip(actual_model, RUN_DIR, scaler=scaler, tokenizer=tokenizer,
                   meta={"init_seed": INIT_SEED, "condition": CONDITION,
                         "phase": 2,
                         "base_run": str(BASE_RUN_DIR),
                         "base_fingerprint": base_fingerprint,
                         "frozen_heads": list(FROZEN_HEADS),
                         "freeze_ab": bool(FREEZE_AB),
                         "loss_w_a": vc.LOSS_W_A, "loss_w_b": vc.LOSS_W_B,
                         "loss_w_c": vc.LOSS_W_C,
                         "num_train_epochs": NUM_EPOCHS,
                         "total_epochs_incl_phase1": 15 + NUM_EPOCHS,
                         "selected_epoch": NUM_EPOCHS,
                         "epoch_selection": "final",
                         "min_eval_loss_epoch": best_epoch_int,
                         "min_eval_loss_value": float(best["eval_loss"]),
                         "train_runtime_s": train_output.metrics.get("train_runtime"),
                         "arm": ARM,
                         "audio_version": AUDIO_VERSION,
                         "frames_version": vc.FEATURES_VERSION})
vc.record_fingerprint(RUN_DIR, "final", actual_model)

[heads] congeladas=nenhuma | head_a:1,025 head_b:3,075 head_c:6,150  (params treináveis por head)
heads restauradas antes do save: 10,250 params entram no checkpoint
2026-08-16 17:56:21,242 - httpx - INFO - _send_single_request() - HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-large-patch14/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-16 17:56:21,401 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-16 17:56:21,553 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-08-16 17:56:21,704 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-08

'93988114ad2403a6'

## 14. Validação do checkpoint

Mesmo padrão da fase 1 — e aqui ela tem uma função extra: **é o teste do portão da seção
13**. Se a restauração de `requires_grad` falhasse, `head_a`/`head_b` não teriam sido
salvas e voltariam do init aleatório com a mesma seed, fazendo
`torch.allclose(loaded, fresh)` acusar. Não remover esta célula.

In [19]:
model_check, scaler_check = vcc.load_run_clip(RUN_DIR, init_seed=INIT_SEED)

mismatches = 0
for key in ["fusion.0.weight", "head_a.weight", "head_b.weight", "head_c.weight"]:
    orig = actual_model.state_dict()[key].cpu()
    loaded = model_check.state_dict()[key].cpu()
    if torch.allclose(orig, loaded):
        print(f"  OK: {key}")
    else:
        print(f"  MISMATCH: {key}")
        mismatches += 1

fresh = vcc.build_model_clip(INIT_SEED, use_bio=USE_BIO, use_viewer_emb=USE_VIEWER_EMB)
differs_from_fresh = not torch.allclose(
    model_check.state_dict()["head_a.weight"].cpu(),
    fresh.state_dict()["head_a.weight"].cpu())

if mismatches == 0 and differs_from_fresh:
    print("\n✓ Checkpoint válido — bate com o modelo treinado e difere do init aleatório")
elif mismatches:
    print(f"\n✗ {mismatches} divergências de peso — algo correu mal no salvamento")
else:
    print("\n✗ Os pesos carregados são idênticos ao init aleatório — não foi salvo")

del model_check, fresh

# Limpeza dos estados por época (seção 10/12) — só depois de RUN_DIR confirmado válido
# acima. Cada arquivo é ~220MB × 15 ≈ 3,3GB; servem só como fonte para o load da
# seção 12, já não têm função depois que a run oficial está gravada e verificada.
if mismatches == 0 and differs_from_fresh:
    import shutil
    n_removed = len(list(EPOCH_STATES_DIR.glob("*.safetensors")))
    shutil.rmtree(EPOCH_STATES_DIR, ignore_errors=True)
    print(f"✓ {n_removed} arquivos de epoch_states/ removidos (~{n_removed * 220}MB liberados)")
else:
    print("⚠ epoch_states/ NÃO removido — checkpoint não passou na validação acima, "
          "mantido para investigação")


2026-08-16 17:56:28,592 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-16 17:56:28,622 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:28,769 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-16 17:56:28,798 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:28,954 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_a

[build_clip_vision] 4 últimas camadas destravadas, 50,384,896 params treináveis
2026-08-16 17:56:29,462 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:29,620 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:29,780 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (3 outputs left out)

[build_clip_vision] 4 últimas camadas destravadas, 50,384,896 params treináveis
2026-08-16 17:56:31,758 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:31,907 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-16 17:56:32,066 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✓ Checkpoint válido — bate com o modelo treinado e difere do init aleatório
✓ 5 arquivos de epoch_states/ removidos (~1100MB liberados)


## 15. Ajuste de limiares da Task C (uma vez, pós-treino)

Mesmo protocolo do braço VideoMAE — limiares ajustados numa metade do val, número
reportável vem da outra metade (ARCH-11 b). `NONE` fixo em 0,5.

In [20]:
threshold_result = vc.tune_task_c_thresholds(
    trainer, val_dataset,
    save_path=str(RUN_DIR / "task_c_thresholds.json"),
)


[Threshold tuning] Gerando predições no conjunto de validação...


2026-08-16 17:57:10,714 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-16 17:57:10,754 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:57:10,867 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-16 17:57:10,868 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:57:10,982 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:57:11,094 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-16 17:57:11,212 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-16 17:57:11,297 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

## 16. Avaliação final e próximos passos

`trainer.evaluate()` é o caminho único de avaliação (OPT-08). `trainer.model` não foi
mutado — `final_metrics` reflete o estado FINAL da época 5 deste braço.

**A comparação que decide, pré-registrada no doc 22 §3:**

1. **Primária — `CLIP_FT5_FREEZEAB` vs `CLIP_FT5_CTRL`.** Único eixo de diferença é o
   freeze. É a única comparação que atribui algo ao freeze.
2. **Secundária — os dois vs `CLIP` (época 15).** Mede o efeito de treinar mais 5 épocas,
   não o do freeze.

As **seis** métricas nas duas comparações, sempre — e C hard também com limiares
ajustados (seção 15), porque foi o tuning que mostrou que a queda de C no WC42 era real e
não artefato de calibração.

⚠ Rodar as duas execuções (`FREEZE_AB = True` e `False`) antes de ler qualquer resultado.
Ler só o braço com freeze e comparar com a época 15 é exatamente o confound de orçamento
que o controle existe para remover.

⚠ Nenhum desfecho autoriza ajustar épocas/LR/pesos e rodar de novo mirando número melhor
(doc 22 §3). Isto é uma medição, não uma busca.

In [21]:
final_metrics = trainer.evaluate()
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

with open(RUN_DIR / "final_metrics.json", "w") as f:
    json.dump({k: (float(v) if isinstance(v, (int, float)) else v)
               for k, v in final_metrics.items()}, f, indent=2)
print(f"\n✓ métricas gravadas em {RUN_DIR / 'final_metrics.json'}")


2026-08-16 17:58:13,355 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-16 17:58:13,396 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:58:13,520 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-16 17:58:13,521 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:58:13,644 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-16 17:58:13,765 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-16 17:58:13,879 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-16 17:58:13,965 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

Training Loss,Validation Loss,Epoch,Task A Icm Hard,Task A Icmnorm Hard,Task A Fmeasure Hard,Task A Icmsoft Soft,Task A Icmsoftnorm Soft,Task A Crossentropy Soft,Task B Icm Hard,Task B Icmnorm Hard,Task B Fmeasure Hard,Task B Icmsoft Soft,Task B Icmsoftnorm Soft,Task B Crossentropy Soft,Task C Icmsoft Soft,Task C Icmsoftnorm Soft,Task C Icm Hard,Task C Icmnorm Hard,Task C Fmeasure Hard,Proxy Task A F1,Proxy Task A Auc,Proxy Task B F1,Proxy Task C F1
0.367292,2.025336,5,0.131431,0.565730,0.710181,-0.013623,0.497606,1.225193,0.040366,0.513924,nan,-0.812552,0.406298,nan,-6.897227,0.139953,-0.625560,0.348586,nan,0.710181,0.768788,0.628662,0.315130


  eval_loss: 2.0253
  eval_proxy_task_a_auc: 0.7688
  eval_proxy_task_a_f1: 0.7102
  eval_proxy_task_b_f1: 0.6287
  eval_proxy_task_c_f1: 0.3151
  eval_task_a_crossentropy_soft: 1.2252
  eval_task_a_fmeasure_hard: 0.7102
  eval_task_a_icm_hard: 0.1314
  eval_task_a_icmnorm_hard: 0.5657
  eval_task_a_icmsoft_soft: -0.0136
  eval_task_a_icmsoftnorm_soft: 0.4976
  eval_task_b_crossentropy_soft: nan
  eval_task_b_fmeasure_hard: nan
  eval_task_b_icm_hard: 0.0404
  eval_task_b_icmnorm_hard: 0.5139
  eval_task_b_icmsoft_soft: -0.8126
  eval_task_b_icmsoftnorm_soft: 0.4063
  eval_task_c_fmeasure_hard: nan
  eval_task_c_icm_hard: -0.6256
  eval_task_c_icmnorm_hard: 0.3486
  eval_task_c_icmsoft_soft: -6.8972
  eval_task_c_icmsoftnorm_soft: 0.1400

✓ métricas gravadas em run_B2_bio_viewer_seed42_CLIP_FT5_FREEZEAB/final_metrics.json
